# PyTorch Basics

In [ ]:
import torch

print(torch.__version__)

2.11.0+cpu


## 1.Understanding Tensors
- It supports both CPUs and GPUs.
- We can create objects of PyTorch’s tensor class
 using the `torch.tensor` function as shown in the following listing.

In [ ]:
import torch

# create a 0D tensor (scalar) from a Python integer
tensor0d = torch.tensor(1)

# create a 1D tensor (vector) from a Python list
tensor1d = torch.tensor([1, 2, 3])

# create a 2D tensor from a nested Python list
tensor2d = torch.tensor([[1, 2],
                         [3, 4]])

# create a 3D tensor from a nested Python list
tensor3d_1 = torch.tensor([[[1, 2], [3, 4]],
                           [[5, 6], [7, 8]]])

**Converting numpy to pytorch**
- copying the Numpy array
- sharing memory using `.from_numpy()`

In [ ]:
import numpy as np
# create a 3D tensor from NumPy array
ary3d = np.array([[[1, 2], [3, 4]],
                  [[5, 6], [7, 8]]])
tensor3d_2 = torch.tensor(ary3d)  # Copies NumPy array
tensor3d_3 = torch.from_numpy(ary3d)  # Shares memory with NumPy array
print(tensor3d_2.dtype)
print(tensor3d_3.dtype)

torch.int64
torch.int64


**1.1 Tensor Datatypes**
1. `int64`
  - 64 bit integer datatype
  - default datatype if no decimal number mentioned

In [ ]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

torch.int64


2. `float32`
  - 32 bit float datatype
  - More preferred since GPU architectures are optimized for 32-bit computations

In [ ]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


- **Conversion of `int64` to `float32`:** using `.to(torch.float32)` as shown below.

In [ ]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

torch.float32


**1.2 Common PyTorch Operations**
1. `.shape`: allows us to access the shape of a tensor

In [ ]:
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])
tensor2d.shape

torch.Size([2, 3])

- It means the tensor has two rows and three columns.

2. `.reshape`: To reshape the tensor into a $3 × 2$ tensor, we can use the `.reshape` method:

In [ ]:
print(tensor2d.reshape(3, 2))
print(tensor2d)

tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])


3. `.view `:Same operation as `.reshape`.

In [ ]:
tensor2d.view(3, 2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

- The subtle difference between `.view()` and `.reshape()` in PyTorch lies in
 their handling of memory layout: `.view()` requires the original data to be contiguous
 and will fail if it isn’t, whereas `.reshape()` will work regardless, copying the data if necessary to ensure the desired shape.

In [ ]:
x=torch.tensor([[[1,2,3,4],[5,6,7,8],[9,10,11,12]],
                [[13,14,15,16],[17,18,19,20],[21,22,23,24]]])
print(x.shape)
x_t = x.transpose(1, 2)
print(x_t)

torch.Size([2, 3, 4])
tensor([[[ 1,  5,  9],
         [ 2,  6, 10],
         [ 3,  7, 11],
         [ 4,  8, 12]],

        [[13, 17, 21],
         [14, 18, 22],
         [15, 19, 23],
         [16, 20, 24]]])


In [ ]:
# Create a tensor
x = torch.randn(2, 3, 4)  # shape: (2, 3, 4)

# Transpose it to make it non-contiguous
x_t = x.transpose(0, 1)  # shape: (3, 2, 4), not contiguous

# Attempt to use .view()
try:
    x_view = x_t.view(6, 4)  # This will fail if x_t is not contiguous
except RuntimeError as e:
    print("view() failed:", e)

# Use .reshape() instead
x_reshape = x_t.reshape(6, 4)  # This works, even if x_t is not contiguous
print("reshape() succeeded:", x_reshape.shape)

view() failed: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape() succeeded: torch.Size([6, 4])


4. `.T`: to transpose a tensor

In [ ]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

5. `.matmul` and `@`:multiply two matrices

In [ ]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [ ]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

## 2. Automatic Differentiation Engine
- Module required: `torch.autograd`


**2.1 Terminologies**
1. `grad`
  - Used to find gradient using `.grad`
  - Used to find gradient $\frac{\partial{y}}{\partial{x}}$ by backpropagating manually from output to input using the syntax `grad(outputs=y,inputs=x)`

In [ ]:
# Create tensors with requires_grad=True
x = torch.tensor(2.0, requires_grad=True)
y = x**2  # y = x^2

# Manually compute gradient dy/dx
grad = torch.autograd.grad(outputs=y, inputs=x, retain_graph=True)
print(grad)

(tensor(4.),)


In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


2. `.backward()`
  - Used for automatic backpropagation using chain rule
  - In this case, we will use `.grad` to display gradient

In [ ]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [ ]:
import torch

# Define a tensor with requires_grad=True
x = torch.tensor(2.0, requires_grad=True)

# Define a function y = x^2
y = x ** 2

# Compute the derivative dy/dx
y.backward()

# Print the gradient
print(x.grad)  # Output: tensor(4.)

tensor(4.)


3. `requires_grad=True`
  - Used to make a variable differentiable
  - By default set to false

In [ ]:
import torch

# Define a tensor with requires_grad=True
x = torch.tensor(2.0)

# Define a function y = x^2
y = x ** 2

# Print the gradient
try:
  # Compute the derivative dy/dx
  y.backward()
  print(x.grad)  # Output: tensor(4.)
except RuntimeError:
  print("Can't find gradient of x")

Can't find gradient of x


In [ ]:
import torch

# Define a tensor with requires_grad=True
x = torch.tensor(2.0,requires_grad=True)

# Define a function y = x^2
y = x ** 2

# Print the gradient
try:
  # Compute the derivative dy/dx
  y.backward()
  print(x.grad)  # Output: tensor(4.)
except RuntimeError:
  print("Can't find gradient of x")

tensor(4.)


4. `retain_graph=True`
  - Used to retain the computation graph
  - By default, it is set to false to save memory
  - Can be applied to both `backward()` and `grad()`

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
y = x ** 2  # y = x^2
y.backward()  # Computes dy/dx = 2x = 4

try:
  y.backward()  # ❌ ERROR: Graph is already deleted!
except RuntimeError:
  print("Graph is deleted,hence can't backpropagate")

Graph is deleted,hence can't backpropagate


In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
y = x ** 2  # y = x^2
y.backward(retain_graph=True)  # Computes dy/dx = 2x = 4

try:
  y.backward()  # ❌ ERROR: Graph is already deleted!
  print("Backpropagated successfully!!")
except RuntimeError:
  print("Graph is deleted,hence can't backpropagate")

Backpropagated successfully!!


## 3.Implementing Neural Networks
- Module required:`torch.nn.Module`
- We can implement neural networks in pytorch in various ways given below

**Simple Implementation**
- Simply listing the layers in `Sequential`.
- **Disadvantage:** No customization in forward pass

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Linear(5, 3),
    nn.Softmax(dim=1)  # Apply softmax along the feature dimension
)
x=torch.rand((3,10))
out=model(x)
print(out)

tensor([[0.2011, 0.2648, 0.5341],
        [0.2090, 0.3241, 0.4669],
        [0.2082, 0.3441, 0.4477]], grad_fn=<SoftmaxBackward0>)


**Subclassing `nn.Module`(Recommended for most cases)**
- **`__init__` constructor :** Used to define the network layers
- **`forward()` :** Used to specify how the layers interact in forward pass.
- No need to specify about `backward` method.

In [ ]:
import torch.nn as nn
import torch

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MyModel()
x = torch.randn(1, 10)
out = model(x)
print(out)

tensor([[ 0.2205, -0.2856]], grad_fn=<AddmmBackward0>)


In [ ]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [ ]:
model = NeuralNetwork(50, 3)

- Displaying the layers of our model

In [ ]:
print(model.layers)

Sequential(
  (0): Linear(in_features=50, out_features=30, bias=True)
  (1): ReLU()
  (2): Linear(in_features=30, out_features=20, bias=True)
  (3): ReLU()
  (4): Linear(in_features=20, out_features=3, bias=True)
)


- Displaying total number of training model parameters.
$$\text{Total number of trainable parameters}=50×30+30+30×20+20+20×3+3=2213$$

In [ ]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [ ]:
X = torch.rand((1, 50))
out = model(X)# Implementing forward function
print(out)

tensor([[-0.0392, -0.1053,  0.1740]], grad_fn=<AddmmBackward0>)


- `.weight`:To find the weights of a layer

In [ ]:
print(model.layers[0].weight)
print(model.layers[0].weight.shape)

Parameter containing:
tensor([[-0.0970, -0.0625, -0.1212,  ..., -0.1204, -0.0556,  0.1095],
        [ 0.0077, -0.0257,  0.1320,  ...,  0.1016,  0.1309,  0.1055],
        [-0.1108,  0.0035,  0.0929,  ...,  0.1119,  0.1361,  0.0683],
        ...,
        [ 0.1174, -0.0666,  0.0223,  ..., -0.0652,  0.0519, -0.1286],
        [ 0.0805,  0.0582, -0.0348,  ...,  0.0201, -0.0178, -0.0249],
        [ 0.0923, -0.1283, -0.0581,  ...,  0.0466,  0.0510, -0.0793]],
       requires_grad=True)
torch.Size([30, 50])


- **`torch.manual_seed()`** :
  - It is used to generate same random numbers with the help of a seed_id given as input in the function
  - Used in random number generating functions like `torch.rand()` and neural networks to control random weight initialization, dropout, shuffling, etc.
  - Enhances reusability of neural networks

In [ ]:
torch.manual_seed(1)
model=NeuralNetwork(50,3)
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.0729, -0.0624, -0.0274,  ...,  0.0416, -0.0408, -0.0155],
        [-0.1360, -0.0674,  0.0767,  ...,  0.0253, -0.0602, -0.0428],
        [ 0.1295, -0.0262,  0.0797,  ...,  0.0457,  0.0863,  0.0952],
        ...,
        [ 0.0723, -0.0578, -0.0877,  ..., -0.0675, -0.0288,  0.0098],
        [-0.0230, -0.0646, -0.0631,  ...,  0.0560, -0.0839, -0.0437],
        [ 0.0006,  0.0852,  0.0799,  ..., -0.1286, -0.0032, -0.0465]],
       requires_grad=True)


In [ ]:
torch.manual_seed(1)
model1=NeuralNetwork(50,3)
print(model1.layers[0].weight)

Parameter containing:
tensor([[ 0.0729, -0.0624, -0.0274,  ...,  0.0416, -0.0408, -0.0155],
        [-0.1360, -0.0674,  0.0767,  ...,  0.0253, -0.0602, -0.0428],
        [ 0.1295, -0.0262,  0.0797,  ...,  0.0457,  0.0863,  0.0952],
        ...,
        [ 0.0723, -0.0578, -0.0877,  ..., -0.0675, -0.0288,  0.0098],
        [-0.0230, -0.0646, -0.0631,  ...,  0.0560, -0.0839, -0.0437],
        [ 0.0006,  0.0852,  0.0799,  ..., -0.1286, -0.0032, -0.0465]],
       requires_grad=True)


In [ ]:
torch.manual_seed(2)
x=torch.rand((1,3))
y=torch.rand((1,3))

In [ ]:
print(x)
print(y)

tensor([[0.6147, 0.3810, 0.6371]])
tensor([[0.4745, 0.7136, 0.6190]])


In [ ]:
torch.manual_seed(2)
x1=torch.rand((1,3))
y1=torch.rand((1,3))
print(x1)
print(y1)

tensor([[0.6147, 0.3810, 0.6371]])
tensor([[0.4745, 0.7136, 0.6190]])


- implementing forward pass in neural network

In [ ]:
torch.manual_seed(1)
model = NeuralNetwork(50, 3)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[ 0.1484, -0.0965,  0.2294]], grad_fn=<AddmmBackward0>)


- **Significance of grad_fn:** It stores the last function used, which is necessary for finding its derivative during backpropagation.

Eg, In the output of the above code the grad_fn used is `<AddmmBackward0>`,in this  case,  it  is  an  Addmm  operation.  Addmm  stands  for  matrix
multiplication (mm) followed by an addition (Add).

- **`no_grad`**
  - It is used to run a neural network without backpropagation
  - Since there is no backpropagation ,there is no construction of computation graph


In [ ]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[ 0.1484, -0.0965,  0.2294]])


## 4. Implementing Dataset and DataLoader classes

### 4.1 `Dataset` Class
Even though we can use our dataset without passing through `Dataset` class , but feeding our data to `Dataset` has the following benifits
- It make the dataset compatible to get loaded in the `DataLoader` class to support batching,shuffling and Multi-threaded data loading.(to be discussed in DataLoader part)
- It provides 3 inbuilt functions
  - `__init__`: to mention feature and labels
  - `__getitem__`: to find item a particular item at an index
  - `__len__`:to find the length of the dataset

We can feed our dataset to `Dataset` class by creating a child class of `Dataset`(here we have create `ToyDataset` class) and passing our dataset as parameters as shown below

In [ ]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

In [ ]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

In [ ]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
train_ds[1] #usage of getitem

(tensor([-0.9000,  2.9000]), tensor(0))

In [ ]:
len(train_ds)

5

### 4.2 `DataLoader` class
It has the following parameters:

1. `dataset`: to load object belonging to `Dataset` class
2. `batch_size`: set size of each batch
3. `shuffle`: To enable suffling,good for training randomness,order can be regenerated using `manual_seed`
4. `num_workers`:To enable multiprocessing of dataset.
5. `drop_last`: To drop the last batch in case of batch size is not divisible by total size, to preserve convergence of neural network

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

In [ ]:
test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

- Printing batches in `train_loader`

In [ ]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


- Since the last batch has less labels than batch size we need to drop it

## 5.Training Model
- Here, were use the following things
1. Optimizers(`torch.optim`)
2. Loss(`torch.nn.functional`)
3. `model.train()` and `model.eval()`
  - This is necessary for components that behave differently during training
and inference, such as dropout or batch normalization layers.
  - `model.train()` activates behaviors like dropout and batch stat updates.
  - `model.eval()` turns them off or uses fixed stats to ensure accurate, stable predictions.
- As discussed earlier, we pass the logits directly into the `cross_entropy` loss function, which will apply the softmax function internally for efficiency and numerical
stability reasons. Then, calling `loss.backward()` will calculate the gradients in the computation graph that PyTorch constructed in the background
-  The `optimizer.step()`
method will use the gradients to update the model parameters to minimize the loss.
In the case of the SGD optimizer, this means multiplying the gradients with the learning rate and adding the scaled negative gradient to the parameters.
- To prevent undesired gradient accumulation, it is important to include
an `optimizer.zero_grad()` call in each update round to reset the gradients to $0$. Otherwise, the gradients will accumulate, which may be undesired.
- Follow the format given below to train and backpropagate using epochs

In [ ]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)

        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/003 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/003 | Train/Val Loss: 0.65
Epoch: 001/003 | Batch 002/003 | Train/Val Loss: 0.42
Epoch: 002/003 | Batch 000/003 | Train/Val Loss: 0.05
Epoch: 002/003 | Batch 001/003 | Train/Val Loss: 0.13
Epoch: 002/003 | Batch 002/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 000/003 | Train/Val Loss: 0.01
Epoch: 003/003 | Batch 001/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 002/003 | Train/Val Loss: 0.02


- You can see how the the cover due to inclusion of last batch, so we will drop it.

In [ ]:
torch.manual_seed(123)# To repeat the same shuffling
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


- Now on running the model we will see a proper convergence

In [ ]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)

        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [ ]:
import torch
import torch.nn as nn

# Set seed for reproducibility
torch.manual_seed(42)

# Dummy input
x = torch.randn(1, 10)  # A single sample with 10 features

# Simple model with Dropout
class DropoutNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
        self.dropout = nn.Dropout(p=0.5)  # Dropout with 50% probability

    def forward(self, x):
        x = self.fc(x)
        x = self.dropout(x)
        return x

model2 = DropoutNet()

# 1. Training mode
model2.train()
print("🔧 Training mode (model.train()):")
for _ in range(3):
    output = model2(x)
    print(output)

# 2. Evaluation mode
model2.eval()
print("\n🧪 Evaluation mode (model.eval()):")
for _ in range(3):
    output = model2(x)
    print(output)


🔧 Training mode (model.train()):
tensor([[ 0.8213,  0.0000,  0.0000, -0.4547,  0.2555]], grad_fn=<MulBackward0>)
tensor([[ 0.8213,  0.0000,  0.0000, -0.4547,  0.0000]], grad_fn=<MulBackward0>)
tensor([[ 0.8213,  0.5006,  0.0000, -0.4547,  0.0000]], grad_fn=<MulBackward0>)

🧪 Evaluation mode (model.eval()):
tensor([[ 0.4107,  0.2503,  0.6282, -0.2274,  0.1277]],
       grad_fn=<AddmmBackward0>)
tensor([[ 0.4107,  0.2503,  0.6282, -0.2274,  0.1277]],
       grad_fn=<AddmmBackward0>)
tensor([[ 0.4107,  0.2503,  0.6282, -0.2274,  0.1277]],
       grad_fn=<AddmmBackward0>)


-  Dropout scales the weights to maintain the same expected value as during training. That's why the values are smaller (e.g., 0.41 instead of 0.82 — roughly half because of 50% dropout).

- We can evaluate metrics(accuracy,precision,recall,f1) at each step using scikit learn package `sklearn.metrics` as shown below(To be discussed later)

- After we have trained the model, we can use it to make predictions:

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [ ]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [ ]:
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [ ]:
predictions == y_train

tensor([True, True, True, True, True])

In [ ]:
torch.sum(predictions == y_train)

tensor(5)

In [ ]:
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [ ]:
compute_accuracy(model, train_loader)

1.0

In [ ]:
compute_accuracy(model, test_loader)

1.0

## 6.Saving and Loading model

- For saving only weights we will use `state_dict()`

In [ ]:
torch.save(model.state_dict(), "model1.pth")

- To load only weight ,not the entire model, we will set `weights_only=True`.

In [ ]:
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model1.pth", weights_only=True))

<All keys matched successfully>

In [ ]:
model.layers[0].weight

Parameter containing:
tensor([[-0.3091,  0.1047],
        [-0.3552,  0.2827],
        [-0.6039,  0.5246],
        [-0.5140, -0.5622],
        [-0.4623,  0.3811],
        [-0.2791,  0.3349],
        [-0.6001, -0.4290],
        [-0.2596, -0.1390],
        [-0.5323,  0.4353],
        [-0.1786,  0.2737],
        [ 0.5025,  0.1076],
        [ 0.1365,  0.7400],
        [-0.3292,  0.2645],
        [-0.3306,  0.5677],
        [ 0.6177, -0.6788],
        [ 0.5121,  0.3864],
        [ 0.2444,  0.3179],
        [ 0.6392, -0.1203],
        [-0.5988, -0.2008],
        [-0.5104,  0.0951],
        [-0.0307, -0.4296],
        [-0.0750,  0.7119],
        [ 0.0188, -0.0559],
        [-0.1308,  0.1919],
        [ 0.6124,  0.4841],
        [ 0.2634,  0.1395],
        [ 0.1408,  0.3752],
        [-0.0796, -0.6639],
        [-0.4392,  0.6110],
        [ 0.0745, -0.5911]], requires_grad=True)

- For saving the entire model

In [ ]:
torch.save(model, "model.pth")  # You didn't do this

- To directly load the model

In [ ]:
loaded_model = torch.load("model.pth", weights_only=False)
loaded_model.eval()  # Set to evaluation mode if needed

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)

In [ ]:
loaded_model.layers[0].weight

Parameter containing:
tensor([[-0.3091,  0.1047],
        [-0.3552,  0.2827],
        [-0.6039,  0.5246],
        [-0.5140, -0.5622],
        [-0.4623,  0.3811],
        [-0.2791,  0.3349],
        [-0.6001, -0.4290],
        [-0.2596, -0.1390],
        [-0.5323,  0.4353],
        [-0.1786,  0.2737],
        [ 0.5025,  0.1076],
        [ 0.1365,  0.7400],
        [-0.3292,  0.2645],
        [-0.3306,  0.5677],
        [ 0.6177, -0.6788],
        [ 0.5121,  0.3864],
        [ 0.2444,  0.3179],
        [ 0.6392, -0.1203],
        [-0.5988, -0.2008],
        [-0.5104,  0.0951],
        [-0.0307, -0.4296],
        [-0.0750,  0.7119],
        [ 0.0188, -0.0559],
        [-0.1308,  0.1919],
        [ 0.6124,  0.4841],
        [ 0.2634,  0.1395],
        [ 0.1408,  0.3752],
        [-0.0796, -0.6639],
        [-0.4392,  0.6110],
        [ 0.0745, -0.5911]], requires_grad=True)